In [ ]:
from dotenv import load_dotenv
import os
load_dotenv()
from openai import OpenAI
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

client = OpenAI()

import pandas as pd
import time
import numpy as np
import json
from pathlib import Path
import re

In [ ]:
# load distribution table
df = pd.read_csv("distribution.csv")


def sample_category(df, column):
    probs = df[column].values
    return np.random.choice(df["Category"], p=probs)

def generate_case(df):
    p_severe = 565 / (1934+565) # Based on the distribution of NACA I-III vs NACA IV-VII in the provided data
    severity = np.random.choice(
        ["NACA_1_3", "NACA_4_7"],
        p=[1 - p_severe, p_severe]
    )

    category = sample_category(df, severity)
    category = [severity, category]
    
    if category[0] == 'NACA_1_3':
        prob_naca_1_2 = 875 / (875 + 1059)
        if np.random.rand() < prob_naca_1_2:
            category[0] = "minor to moderate disturbance" # NACA I - II
        else:
            category[0] = "severe but not life-threatening disturbance" # NACA III
    if category[0] == 'NACA_4_7':
        prob_rea = 40 / 565 # Based on the resuscitation or death cases in the provided data
        if np.random.rand() >= prob_rea:
            category[0] = "severe and potential life-threatening disturbance" # NACA IV - V
        else:
            category[0] = "Resuscitation or Death" # NACA VI - VII

    return category[0], category[1]


def get_cases(i, df):
    res = []
    for _ in range(i):
        case = generate_case(df)
        res.append(case)
    return res


def build_user_prompt_eng(case):
    return f"""
Create a realistic emergency scenario for the German emergency medical services (EMS).
Be creative when choosing the scenario.

Structure:

Current event:
- What happened?
- How was the emergency noticed?
- Timeline / course of events

On-scene situation:
- Environment (apartment, street, workplace, etc.)
- Number of people involved
- Hazards or special circumstances
- Behavior of relatives / witnesses / first responders

Patient history:
- Relevant pre-existing conditions
- Medications (if known)
- Known risk factors

Write precisely and realistically, approx. 150–200 words.

The scenario is a {case[1]}. The severity level is a {case[0]}, derived from the NACA score.
Be creative in selecting the scenario, but avoid clichés. It should be a realistic but not everyday scenario that reflects the diversity of EMS emergencies in Germany.
""".strip()


def build_user_prompt(case):
    return f"""
Erstelle ein realistisches Notfall-Szenario für den Rettungsdienst in Deutschland.
Seit kreativ bei der Auswahl des Szenarios. 

Struktur:

Aktuelles Ereignis:
- Was ist passiert?
- Wie wurde der Notfall bemerkt?
- Zeitlicher Verlauf

Situation vor Ort:
- Umgebung (Wohnung, Straße, Arbeitsplatz etc.)
- Anzahl beteiligter Personen
- Gefahren oder Besonderheiten
- Verhalten der Angehöriger/ Zeugen/ Ersthelfer

Patientenvorgeschichte:
- relevante Vorerkrankungen
- Medikamente (falls bekannt)
- bekannte Risiken

Schreibe präzise und realistisch, ca. 150–200 Wörter.

Das Szenario ist eine {case[1]}. Der Schweregrad ist eine {case[0]}, abgeleitet aus dem NACA-Score.
Sei kreativ bei der Auswahl des Szenarios, aber vermeide Klischees. Es soll ein realistisches, aber nicht alltägliches Szenario sein, das die Vielfalt der Notfälle im Rettungsdienst widerspiegelt. 
""".strip()


def write_batch_input(cases, language='ger',  out_path="batch_input_emergency_text.jsonl"):
    
        
    if language == 'ger':
        system_prompt = (
        "Du erstellst realistische Notfallszenarien für Training im Rettungsdienst. "
        "Nutze klare, sachliche Sprache wie in einer Einsatzdokumentation. "
        "Keine echten personenbezogenen Daten."
    )
    else:
        system_prompt = (
            "You create realistic emergency scenarios for training in the German emergency medical services (EMS). "
            "Use clear, factual language as in an incident report. "
            "No real personal data."
        )

    out_path = Path(out_path)

    with out_path.open("w", encoding="utf-8") as f:
        for i, case in enumerate(cases):
            if language == 'ger':
                content = build_user_prompt(case)
            else:
                content = build_user_prompt_eng(case)
            req = {
                "custom_id": f"case_{i}",
                "method": "POST",
                "url": "/v1/responses",
                "body": {
                    "model": "gpt-5.2",
                    "input": [
                        {"role": "system", "content": system_prompt},
                        {"role": "user", "content": content},
                    ],
                    "temperature": 1
                }
            }
            f.write(json.dumps(req, ensure_ascii=False) + "\n")

    return str(out_path)


def download_and_parse_batch(batch_id: str, out_jsonl="batch_results.jsonl"):
    b = client.batches.retrieve(batch_id)
    if b.status != "completed":
        raise RuntimeError(f"Batch not completed. Current status: {b.status}")

    content = client.files.content(b.output_file_id)
    Path(out_jsonl).write_bytes(content.read())

    results = {}
    with open(out_jsonl, encoding="utf-8") as f:
        for line in f:
            item = json.loads(line)

            custom_id = item["custom_id"]

            response_obj = (item.get("response", {}) or {}).get("body", {}) or {}
            output = response_obj.get("output", []) or []

            text_parts = []
            for out in output:
                for c in out.get("content", []):
                    if c.get("type") == "output_text":
                        text_parts.append(c.get("text", ""))

            results[custom_id] = "\n".join([t for t in text_parts if t]).strip()

    return results
    

def save_cases(case_dict, folder="cases"):
    folder_path = Path(folder)
    folder_path.mkdir(exist_ok=True)

    # vorhandene case_XX Dateien scannen
    existing_numbers = []

    for file in folder_path.glob("case_*.txt"):
        match = re.search(r"case_(\d+)", file.stem)
        if match:
            existing_numbers.append(int(match.group(1)))

    start_index = max(existing_numbers) + 1 if existing_numbers else 0

    # neue Fälle speichern
    for i, (_, text) in enumerate(case_dict.items()):
        file_path = folder_path / f"case_{start_index + i}.txt"
        file_path.write_text(text.strip() + "\n", encoding="utf-8")

    print(f"{len(case_dict)} Fälle gespeichert ab case_{start_index}")

In [ ]:
cases = get_cases(30, df)

path = write_batch_input(cases)

# Upload batch file
batch_file = client.files.create(
    file=open(path, "rb"),
    purpose="batch"
)

# Start Batch
batch = client.batches.create(
    input_file_id=batch_file.id,
    endpoint="/v1/responses",
    completion_window="24h"
)

print("Batch ID:", batch.id)


In [ ]:
# Check batch status until completion
while True:
    status = client.batches.retrieve(batch.id)
    print("Status:", status.status)

    if status.status in ["completed", "failed", "cancelled", "expired"]:
        break

    time.sleep(10)

In [ ]:
res = download_and_parse_batch(batch.id)
save_cases(res)